# Stable linear regression and probability

**P1 Core · D2 Independent · 100 minutes**

In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
rng = np.random.default_rng(20260718)
x1 = rng.normal(size=250)
x2 = x1 + 1e-7 * rng.normal(size=250)
X = np.column_stack([np.ones(len(x1)), x1, x2])
y = 1.5 + 3.0*x1 - 2.0*x2 + 0.1*rng.normal(size=len(x1))
condition = np.linalg.cond(X)
condition

## Task

Fit coefficients without an explicit inverse and return coefficients, predictions, residuals, MAE, RMSE, and Gaussian negative log-likelihood up to an additive constant.

In [ ]:
def stable_fit(X: np.ndarray, y: np.ndarray) -> dict:
    beta, _, rank, singular = np.linalg.lstsq(X, y, rcond=None)
    prediction = X @ beta
    residual = y - prediction
    variance = np.mean(residual**2)
    return {'beta': beta, 'prediction': prediction, 'residual': residual,
            'rank': int(rank), 'singular': singular,
            'mae': mean_absolute_error(y, prediction),
            'rmse': mean_squared_error(y, prediction)**0.5,
            'gaussian_nll': 0.5*len(y)*np.log(variance) + 0.5*np.sum(residual**2)/variance}

In [ ]:
result = stable_fit(X, y)
assert condition > 1e6
assert result['rank'] == X.shape[1]
assert result['rmse'] < 0.12
ridge = Ridge(alpha=1.0, fit_intercept=False).fit(X, y)
assert np.linalg.norm(ridge.coef_) < np.linalg.norm(result['beta'])
{k: result[k] for k in ('rank','mae','rmse','gaussian_nll')}

## Transfer

Use a chronological air-quality holdout. Compare stable least squares and Ridge, inspect residuals by time/prediction, and state which Gaussian and independence assumptions fail.